# Handoffs

*Notebook 12*

Route tasks to the right specialist automatically using the handoff pattern.


---

## 🔧 Setup

In [ ]:
from pathlib import Path
from dotenv import load_dotenv
load_dotenv(dotenv_path=Path("..") / ".env")

from agents import Agent, Runner, handoff

MODEL = "gpt-5-mini"


print("✅ Ready!")

---

## 🎯 The Problem

A single support agent has to juggle billing, technical support, and refunds.

Handoffs let a triage agent route each request to focused instructions and tone.

---

## 🔄 Part 1: How Handoffs Work

A handoff transfers responsibility to another agent.

The SDK exposes each target as a tool, such as `transfer_to_billingagent`.

When called, the target becomes the active agent and receives the conversation history by default.

In this graph, specialists have no handoffs, so triage does not resume automatically.

```
User message
     ↓
Triage Agent   ← decides who should handle this
     ↓ handoff
Specialist Agent  ← handles the request and responds
```

---

## 🏥 Part 2: A Simple Triage System

We'll build a customer support system with a triage agent and three specialists (billing, technical support, and refunds).

The agent `name` becomes part of the auto-generated handoff tool name (e.g., `BillingAgent` → `transfer_to_billingagent`).

Names with spaces get underscores (`"Billing Agent"` → `transfer_to_billing_agent`), so pick names that read well as routing destinations.

In [ ]:
billing_instructions = (
    "You are a billing specialist. You help with:\n"
    "- Invoice questions and payment history\n"
    "- Subscription changes and cancellations\n"
    "- Billing errors and duplicate charges\n"
    "Be empathetic and solution-focused. Always offer a clear next step."
)

billing_agent = Agent(
    name="BillingAgent",
    instructions=billing_instructions,
    model=MODEL
)

tech_instructions = (
    "You are a technical support specialist. You help with:\n"
    "- Setup and installation issues\n"
    "- Bug reports and error messages\n"
    "- Feature questions and how-to guidance\n"
    "Be precise and step-by-step in your responses."
)

tech_agent = Agent(
    name="TechSupportAgent",
    instructions=tech_instructions,
    model=MODEL
)

refunds_instructions = (
    "You are a refunds specialist. You help with:\n"
    "- Processing refund requests\n"
    "- Refund status and timelines\n"
    "- Eligibility questions and exceptions\n"
    "Be clear about timelines and set accurate expectations."
)

refunds_agent = Agent(
    name="RefundsAgent",
    instructions=refunds_instructions,
    model=MODEL
)

# --------------------------------------------------------------
print("✅ Specialist agents defined")

#### Define the Triage Agent

In [ ]:
triage_instructions = (
    "You are a customer support triage agent. Your only job is to route requests.\n\n"
    "- Billing questions (invoices, payments, subscriptions) → hand off to BillingAgent\n"
    "- Technical questions (bugs, setup, errors, features) → hand off to TechSupportAgent\n"
    "- Refund requests and refund status → hand off to RefundsAgent\n\n"
    "Do not answer questions yourself. Always hand off to the right specialist."
)

triage_agent = Agent(
    name="TriageAgent",
    instructions=triage_instructions,
    model=MODEL,
    handoffs=[billing_agent, tech_agent, refunds_agent]
)

# --------------------------------------------------------------
print("✅ Triage agent ready")
print("   Handoffs: BillingAgent, TechSupportAgent, RefundsAgent")

### 💡 Key Takeaway

The SDK auto-creates `transfer_to_billingagent`, `transfer_to_techsupportagent`, and `transfer_to_refundsagent` tools from the agent names.

Routing is prompt-driven, not enforced.

Clear, narrow triage instructions guide the agent to hand off instead of answering directly.

---

## 🧪 Part 3: See the Handoffs in Action

`result.last_agent.name` returns the agent that produced the final response, useful for debugging routing and verifying triage decisions.

In [ ]:
# Labeled routing cases: each message with the specialist we expect to handle it
test_cases = [
    {"message": "I was charged twice for my subscription this month.", "expected": "BillingAgent"},
    {"message": "My app crashes when I try to export a PDF.", "expected": "TechSupportAgent"},
    {"message": "I can't get the desktop app to install on Windows.", "expected": "TechSupportAgent"},
    {"message": "I'd like to request a refund for last month's charge.", "expected": "RefundsAgent"},
]

# --------------------------------------------------------------
print(f"✅ {len(test_cases)} routing cases ready")

#### Run All Test Messages

In [ ]:
print("🔄 HANDOFF DEMO")
print("=" * 60)

# Routing is model-driven: these checks reveal what happened in this run
passed = 0
for case in test_cases:
    result = await Runner.run(triage_agent, input=case["message"])
    actual = result.last_agent.name
    ok = actual == case["expected"]
    passed += ok

    print(f"\n📨 Message: {case['message']}")
    print(f"🤖 Expected: {case['expected']} | Actual: {actual} | {'✅ pass' if ok else '❌ FAIL'}")
    print(f"Response: {result.final_output}")
    print("-" * 60)

print(f"\nRouting: {passed}/{len(test_cases)} matched the expected specialist this run")

---

## ⚙️ Part 4: Customizing Handoffs

Use the `handoff()` function for more control over how each handoff is described to the triage agent.

A custom description tells it exactly when to use each specialist.

In [ ]:
triage_v2_instructions = (
    "You are a customer support triage agent.\n"
    "Use a handoff only when its description clearly matches.\n"
    "If none clearly matches, ask one clarifying question instead of guessing.\n"
    "Do not answer support questions yourself."
)

triage_agent_v2 = Agent(
    name="TriageAgentV2",
    instructions=triage_v2_instructions,
    model=MODEL,
    handoffs=[
        handoff(
            billing_agent,
            tool_description_override="Transfer to billing for invoice, payment, or subscription questions."
        ),
        handoff(
            tech_agent,
            tool_description_override="Transfer to tech support for bugs, errors, installation, or feature questions."
        ),
        handoff(
            refunds_agent,
            tool_description_override="Transfer to refunds for refund requests, refund status, and eligibility questions."
        )
    ]
)

# --------------------------------------------------------------
print("✅ TriageAgentV2 ready with custom handoff descriptions")

#### Compare Routing on an Ambiguous Message

"Account settings" matches none of the specialist descriptions.

v1 is told to always hand off, so it may force a route.

v2 is allowed to ask for clarification instead.

This is one stochastic comparison, not proof that v2 is always better.

In [ ]:
ambiguous_message = "I need help with my account settings."

print("🔄 BEFORE/AFTER COMPARISON")
print("=" * 60)

result_v1 = await Runner.run(triage_agent, input=ambiguous_message)

print(f"📨 Message: {ambiguous_message}")
print(f"🤖 v1 (simple handoffs) handled by: {result_v1.last_agent.name}")

result_v2 = await Runner.run(triage_agent_v2, input=ambiguous_message)

print(f"🤖 v2 (custom descriptions) handled by: {result_v2.last_agent.name}")
print(f"    v2 said: {result_v2.final_output}")
print("=" * 60)

### 💡 Key Takeaway

Custom `tool_description_override` gives the triage agent more explicit routing criteria.

Use plain `handoffs=[...]` when the agent names already make the routing obvious.

Reach for `handoff(..., tool_description_override=...)` when requests could plausibly fit two specialists.

---

## 💪 Practice Exercises

### Exercise 1: Four-Way Routing

*Covers: handoffs, extending a triage routing system*

Add a fourth specialist (an FAQ agent) and update the triage agent to route to all four.

In [ ]:
# --------------------------------------------------------------
# 💪 Exercise 1: Four-Way Routing
# --------------------------------------------------------------
# Objective: Add an FAQ agent and update triage to route to all four.

# TODO 1: Create an FAQ agent named FAQAgent with instructions to answer
#          general questions about pricing, features, and company policies

# TODO 2: Create a new triage agent with handoffs to:
#          billing_agent, tech_agent, refunds_agent, and your new faq_agent
#          Use handoff() with tool_description_override for all four

# TODO 3: Test with labeled cases (message, expected handler) and compare
#          result.last_agent.name to the expected agent, printing pass/fail:
#          - "What is your pricing?"        -> FAQAgent
#          - "The app shows an error when I try to log in" -> TechSupportAgent
#          - "When will my refund arrive?"  -> RefundsAgent

# --- Write your code below this line ---

### Exercise 2: Domain-Specific Specialist

*Covers: handoffs, building a domain-specific routing system*

Build a different triage system for a different domain: a recipe assistant that routes to a vegetarian specialist or a meat specialist.

In [ ]:
# --------------------------------------------------------------
# 💪 Exercise 2: Recipe Assistant
# --------------------------------------------------------------
# Objective: Build a triage system for a different domain.

# TODO 1: Create a VegetarianChef agent with relevant instructions

# TODO 2: Create a MeatChef agent with relevant instructions

# TODO 3: Create a RecipeTriage agent that routes to the right chef

# TODO 4: Test with: "How do I make a beef burger?"
#         and: "What's a good lentil dish?"
#         Print result.last_agent.name and result.final_output

# --- Write your code below this line ---

---

## 🎯 Key Takeaways

**Handoffs change the active agent:**

- The target agent takes over and sees the full conversation history by default

- If the specialist needs more information, it can ask the user directly

- In a graph where the target has no handoffs, control does not return automatically
<br>
<br>

**Two ways to configure handoffs:**

- Bare agents (`handoffs=[agent]`) create default handoff names and descriptions

- `handoff(agent, tool_description_override="...")` adds routing criteria, but does not enforce them

- Routing stays model-driven, so test labeled cases and check the result
<br>
<br>

**Verify who handled the request:**

- `result.last_agent.name` identifies who produced the final response

- Check the tracing dashboard (Lesson 23) to see the full handoff chain

---

## 📍 Next Step

**Notebook 13: Agents as Tools**  

Learn when to call an agent like a function instead of handing off to it, and why the distinction matters.

---

##### 🔧 [Troubleshooting Guide](https://github.com/barrettscott/openai-agents/blob/main/TROUBLESHOOTING.md#lesson-12-handoffs)

---